# Data Profiling Summary

Complete transparency on dataset structure, quality, and pipeline outputs. Auto-generated for stakeholder visibility and reproducibility assurance.

## Raw Datasets (De-Risk Validation)

In [1]:
import json
from pathlib import Path
derisk_report = json.loads(Path('reports/derisk_inspection_report.json').read_text())
# Raw dataset validation summary
print(json.dumps(derisk_report, indent=2))

{
  "run_timestamp_utc": "2026-05-21T10:00:28.498452+00:00",
  "manifest_path": "config/dataset_manifest.yaml",
  "dataset_count": 5,
  "datasets_accessible": 5,
  "datasets_inaccessible": 0,
  "datasets": {
    "unitreerobotics/G1_WBT_Inspire_Pickup_Pillow_MainCamOnly": {
      "sha": "24e3e4d88a5020bdb4b3046ec09b09dc56f8d1f1",
      "last_modified": "2026-03-27T12:03:23+00:00",
      "files_count": 34,
      "accessible": true,
      "error": null
    },
    "unitreerobotics/G1_WBT_Inspire_Put_Clothes_into_Washing_Machine_MainCamOnly": {
      "sha": "c0a5fb0992a0f2a2b9df3493d27c2d670a4b1c36",
      "last_modified": "2026-03-27T12:03:44+00:00",
      "files_count": 24,
      "accessible": true,
      "error": null
    },
    "unitreerobotics/G1_WBT_Brainco_Collect_Plates_Into_Dishwasher": {
      "sha": "16c01dbfcb2159783ea575acd42d1cec9b69e311",
      "last_modified": "2026-03-27T12:04:41+00:00",
      "files_count": 114,
      "accessible": true,
      "error": null
    },
    "uni

## Module 1: Humanoid Capability Summary

### Schema

| Column | Type | Non-Null | Example |
|--------|------|----------|----------|
| task_category | String | True | unclassified |
| n_episodes | UInt64 | True | 2359 |
| cycle_time_p50 | Float64 | True | 67.4 |
| cycle_time_p95 | Float64 | True | 139.6 |
| cycle_time_mean | Float64 | True | 61.404790165324286 |
| cycle_time_std | Float64 | True | 41.78692018853171 |
| reach_mean_meters | Float64 | True | 0.275694821970724 |
| reach_max_meters | Float64 | True | 1.4568268687729193 |
| energy_proxy_mean | Float64 | True | 10671.567706476542 |
| success_rate | Float64 | True | 1.0 |
| insufficient_sample | Boolean | True | False |


### Descriptive Statistics

In [2]:
import polars as pl
summary = pl.read_parquet('data/processed/humanoid_capabilities_summary.parquet')
print(summary.describe())

shape: (9, 12)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ statistic ┆ task_cate ┆ n_episode ┆ cycle_tim ┆ … ┆ reach_max ┆ energy_pr ┆ success_r ┆ insuffic │
│ ---       ┆ gory      ┆ s         ┆ e_p50     ┆   ┆ _meters   ┆ oxy_mean  ┆ ate       ┆ ient_sam │
│ str       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ple      │
│           ┆ str       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ count     ┆ 4         ┆ 4.0       ┆ 4.0       ┆ … ┆ 3.0       ┆ 3.0       ┆ 4.0       ┆ 4.0      │
│ null_coun ┆ 0         ┆ 0.0       ┆ 0.0       ┆ … ┆ 1.0       ┆ 1.0       ┆ 0.0       ┆ 0.0      │
│ t         ┆           ┆           ┆           ┆   ┆           ┆           

### Data Quality

In [3]:
# Missing values per column
missing = {col: summary[col].null_count() for col in summary.columns}
print('Missing values:', missing)
print('Categories with insufficient_sample=True:', summary.filter(pl.col('insufficient_sample') == True).shape[0])

Missing values: {'task_category': 0, 'n_episodes': 0, 'cycle_time_p50': 0, 'cycle_time_p95': 0, 'cycle_time_mean': 0, 'cycle_time_std': 0, 'reach_mean_meters': 1, 'reach_max_meters': 1, 'energy_proxy_mean': 1, 'success_rate': 0, 'insufficient_sample': 0}
Categories with insufficient_sample=True: 0


### Full Summary Table

In [4]:
summary

task_category,n_episodes,cycle_time_p50,cycle_time_p95,cycle_time_mean,cycle_time_std,reach_mean_meters,reach_max_meters,energy_proxy_mean,success_rate,insufficient_sample
str,u64,f64,f64,f64,f64,f64,f64,f64,f64,bool
"""bimanual_handling""",525,78.7,88.2,78.769143,7.565853,0.520711,1.456827,10236.849166,1.0,false
"""pick_medium_object""",2359,67.4,139.6,61.40479,41.78692,0.275695,1.456827,10671.567706,1.0,false
"""place_general""",1750,73.0,153.2,73.7988,41.860308,0.275695,1.456827,10671.567706,1.0,false
"""transport_short""",757,31.3,192.8,72.177807,62.953537,null,null,null,1.0,false


## Module 1: Per-Episode Features

In [5]:
import polars as pl
per_episode = pl.read_parquet('data/processed/humanoid_capabilities_per_episode.parquet')
print(f'Shape: {per_episode.shape}')
print(f'Columns: {per_episode.columns}')
print('\nFirst 5 rows:')
per_episode.head(5)

Shape: (2359, 12)
Columns: ['episode_id', 'cycle_time_seconds', 'reach_meters_estimate', 'energy_proxy_joint_integral', 'n_frames', 'success_inferred', 'task_description', 'dataset_repo_id', 'phase', 'task_category_source', 'task_categories', 'task_category']

First 5 rows:


episode_id,cycle_time_seconds,reach_meters_estimate,energy_proxy_joint_integral,n_frames,success_inferred,task_description,dataset_repo_id,phase,task_category_source,task_categories,task_category
i64,f64,f64,f64,i64,bool,str,str,i32,str,list[str],str
0,26.1,null,null,261,true,"""task""","""unitreerobotics/G1_WBT_Inspire…",1,"""pick_medium""","[""pick_medium_object""]","""pick_medium_object"""
1,24.2,null,null,242,true,"""task""","""unitreerobotics/G1_WBT_Inspire…",1,"""pick_medium""","[""pick_medium_object""]","""pick_medium_object"""
2,24.6,null,null,246,true,"""task""","""unitreerobotics/G1_WBT_Inspire…",1,"""pick_medium""","[""pick_medium_object""]","""pick_medium_object"""
3,23.0,null,null,230,true,"""task""","""unitreerobotics/G1_WBT_Inspire…",1,"""pick_medium""","[""pick_medium_object""]","""pick_medium_object"""
4,23.3,null,null,233,true,"""task""","""unitreerobotics/G1_WBT_Inspire…",1,"""pick_medium""","[""pick_medium_object""]","""pick_medium_object"""


---

**Generated by:** `warehouse_humanoid_tco.analysis.profile_outputs`
**Purpose:** Transparency, reproducibility, stakeholder visibility